In [1]:
from neo4j import GraphDatabase
from igraph import Graph
import json
from datetime import datetime, timedelta
from random import randint

In [2]:
g = GraphDatabase.driver('bolt://localhost:7687', auth=('neo4j', '123'))

In [3]:
def get_all_edge():
    with g.session() as session:
        query = ("""
        MATCH (s)-[r]->(t)
        RETURN s, t, r
        """)
        re = session.run(query).values()
    return re

def get_all_node():
    with g.session() as session:
        query = ("""
        MATCH (n) return n
        """)
        return session.run(query).values()

def date_time(d):
    t = '/'.join([str(d.year), str(d.month), str(d.day)])
    return datetime.strptime(t, '%Y/%m/%d')

edges = get_all_edge()
nodes = get_all_node()

In [8]:
graph = Graph(directed = False)

def gen_nodes(nodes):
    for i in range(len(nodes)):
        v = nodes[i][0]
        graph.add_vertex(name=v['name'])
        graph.vs[i]['age_group'] = v['age_group']
        graph.vs[i]['full_name'] = v['full_name']
        graph.vs[i]['label'] = v.labels
        graph.vs[i]['onset_date'] = date_time(v['onset_date'])
        graph.vs[i]['announce_date'] = date_time(v['announce_date'])
        graph.vs[i]['pagerank'] = 2

def gen_edge(edges):
    for i in range(len(edges)):
        start_node = edges[i][0]
        end_node = edges[i][1]
        r_type = edges[i][2].type
        graph.add_edge(start_node['name'], end_node['name'], weight = 1, r_type = r_type)
        
        
gen_nodes(nodes)
gen_edge(edges)

In [17]:
def gen_node_color(graph, mode='age_group'):
    mode_list = ['age_group', '']
    
    

def gen_json(graph, color_mode='default' ,path="../visualization/data.json"):
    gr = {'nodes':[], 'edges':[]}
    pos = graph.layout_fruchterman_reingold()
    for v, t in zip(graph.vs, pos):
        vertex = {}
        vertex['id'] = v.index
        vertex['label'] = v['name']
        vertex['full_name'] = v['full_name']
        vertex['x'] = t[0]
        vertex['y'] = t[1]
        vertex['age_group'] = v['age_group']
        vertex['onset_date'] = v['onset_date'].strftime('%d/%m/%Y')
        vertex['announce_date'] = v['announce_date'].strftime('%d/%m/%Y')
        vertex['pagerank'] = v.pagerank()
        vertex['size'] = 700*vertex['pagerank']
        vertex['_color'] = '#fc5148'
        gr['nodes'].append(vertex)
    for e in graph.es:
        edge = {}
        edge['id'] = e.index
        edge['source'] = e.source
        edge['target'] = e.target
        edge['weight'] = e['weight']
        edge['type'] = e['r_type']
        edge['size'] = 20*e['weight']
#         edge['type'] = 'curvedArrow'
        edge['_color'] = '#34c0eb'
        gr['edges'].append(edge)
    try:
        with open(path,'w+', encoding='utf8') as f:
            json.dump(gr, f)
        return True
    except:
        return False

gen_json(graph)

True

In [101]:
def gen_weight(edge, date):
    

<Layout with 389 vertices and 2 dimensions>

In [ ]:
graph.layout_grid_fruchterman_reingold()